# Ocean Emulator UNet: Forward Ocean Heat Content Rollout with ACCESS-OM2

This notebook trains a clean **autoregressive forward emulator** for ocean heat
content using ACCESS-OM2 data. Given a short window of past OHC states and the
current surface heat flux as forcing, the model predicts the next OHC state.
Chaining this one-step predictor forward produces a continuous OHC rollout.

The model used here is deliberately simple: a deterministic `ForwardUNet` wrapper
around the existing `UNet` / `PartialConv2d` architecture from `Emulator.py`.
The training path is baseline-only.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: import dependencies and define paths/device for the full
# forward-emulator workflow. Extends the import list from AutoEncoder_om2.ipynb
# with petpipe.modifications and petpipe.branching, used below to build the
# sliding-window / state-forcing-split pipeline.
# Outcome: pipeline/data/model/plot modules are loaded and local utils are importable.
# -----------------------------------------------------------------------------
# System and path handling
import sys
import functools
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
import xarray as xr

# PyEarthTools pipeline
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
import lightning as L

# Visualization
import matplotlib.pyplot as plt

# Configure device and random seed
torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Source data and code directories
userbase = "/g/data/nm47/txs156/"
# Taimoor's repobase:
repobase = "/home/156/txs156/uom/OM2-emulator/"
# Ryan's repobase:
# repobase = "/home/561/rmh561/ML/OM2-emulator/"
# Navid's repobase:
# repobase = "/home/552/nc3020/gdata/OM2-emulator/"

# Add the src directory to the Python path (relative to notebook location)
src_path = Path(repobase) / "src"
sys.path.insert(0, str(src_path))

# Import local OM2 emulator modules
# Same building blocks as AutoEncoder_om2.ipynb -- the UNet and PartialConv2d
# architecture are kept as-is. AutoEncoder is no longer used: the forward emulator
# is not a reconstruction model, so there is no single-frame autoencoding step.
from Data import ACCESS_OHC, build_normalisation, make_fast_dl
from Emulator import LightningWrapper, PartialConv2d, UNet
import warnings


## Data & normalisation

Unchanged from `AutoEncoder_om2.ipynb`. The normalisation strategy and the OHC/flux
variable list are independent of whether the model reconstructs one frame or predicts
the next one, so this step does not need to change.

In [ ]:
# Setup the normalisation
time_start = '2000-01'
train_end = "2015-03"
val_start = "2015-04"
time_end = '2018-12'
time_interval = '1MS'
norm_variables = ["ocean_heat_content_2d", "total_surface_heat_flx"]
norm_strat = "Spatial_climatology"
datapath = userbase + "OM2-emulator/data/1deg_ocean_heat_emulator_data.nc"

mask, normalisation = build_normalisation(datapath, norm_strat, norm_variables, time_window = dict(start=time_start, end=time_end, freq=time_interval),\
                                          train_end = train_end, mask = True)
mean = normalisation._initialisation["mean"]
deviation = normalisation._initialisation["deviation"]


In [ ]:
# Set up accessor for the ACCESS data
ACCESS_OHC_accessor = ACCESS_OHC(
    ["area_t", "ocean_heat_content_2d", "total_surface_heat_flx"],
    root=userbase + "OM2-emulator/data/",
)


## Setup the forward-emulation PET pipeline

This is the core structural change from `AutoEncoder_om2.ipynb`. The original pipeline
iterates over single dates and emits one `(OHC, flux)` frame per sample, which is fine
for reconstruction but has no notion of "predict the next state given recent history and
forcing."

A hand-rolled windowing wrapper around `DateRange` was the first instinct here, but PET
already has purpose-built, tested classes for exactly this pattern, in
`pyearthtools.pipeline.modifications` (`petpipe.modifications`):

- **`TemporalWindow`** — built specifically, per its own docstring, "to provide the
  ability to perform sequence-to-sequence modelling from a data accessor or pipeline
  that was designed to produce single time steps." Given `prior_indexes` and
  `posterior_indexes` (each multiplied by a `timedelta` and applied to the queried
  reference date), it returns a `(prior, posterior)` tuple directly. This is the natural
  fit for "two past states in, one future state out," matching the
  `Φ̃_{t+(n-1)Δt}, Φ̃_{t+nΔt} → Φ̃_{t+(n+1)Δt}` recurrence in Samudra (their Eq. 1).
- **`SequenceRetrieval` / `TemporalRetrieval`** — the more general, lower-level
  mechanism `TemporalWindow` is built on. Useful if a non-contiguous or asymmetric
  sampling pattern is ever needed, but `TemporalWindow`'s narrower interface is the
  better fit for this fixed two-in-one-out window and is used below.

For separating OHC (a *state* to be windowed and predicted) from surface flux (a
*forcing* that is supplied, never reconstructed), PET's `petpipe.branching` module
provides `PipelineBranchPoint`: it forks the same upstream sample down independent
sub-pipelines and returns their results as a tuple, in declared order. This replaces
what would otherwise be a manual "predict everything, then mask flux out of the loss"
workaround with two clean, independently-composed branches:

1. a **state branch**: `SelectDataset(["ocean_heat_content_2d"])` &rarr; `TemporalWindow`
2. a **forcing branch**: `SelectDataset(["total_surface_heat_flx"])`, supplying only the
   forcing at the timesteps the rollout actually needs

Each branch is itself an ordinary PET pipeline, so the existing transforms, the
normalisation step, `FillNan`, and `ToNumpy` conversion are reused unchanged within each
branch — nothing about those steps needs to be rewritten, only re-arranged around the
new branch structure.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: define the two re-usable sub-pipeline step sequences (state, forcing)
# that will be forked from the same upstream accessor inside a PipelineBranchPoint.
# Outcome: `state_steps` and `forcing_steps` are tuples of PET pipeline steps, each
# independently valid as the body of a Pipeline(...) call.
# -----------------------------------------------------------------------------

# Both branches start from the same coordinate clean-up as AutoEncoder_om2.ipynb,
# then diverge at the variable-selection step.
_shared_coord_drop = petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t'], ignore_missing=True)
_shared_var_drop = petdata.transforms.variables.Drop(['area_t'])

# State branch: OHC only. This is the field that is windowed in time (two past
# states as input context, matching Samudra's 2-input/2-output recurrence) and is
# the model's prediction target.
state_steps = (
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(["ocean_heat_content_2d"]),
    petpipe.operations.xarray.Sort(order=["ocean_heat_content_2d"], strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t'], ignore_missing=True),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)

# Forcing branch: surface heat flux only. This is a known boundary condition the
# emulator conditions on -- per Subel & Zanna (2024), surface forcing (alongside wind
# stress, not present in this 2D OHC-only dataset) is what prevents an emulator from
# drifting in the absence of any external driving signal. It is never reconstructed,
# so it is not windowed across (t-dt, t) the way the state branch is -- only the
# forcing at the *target* timestep is needed to predict that timestep's OHC.
forcing_steps = (
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(["total_surface_heat_flx"]),
    petpipe.operations.xarray.Sort(order=["total_surface_heat_flx"], strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t'], ignore_missing=True),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: define the rollout time step and the TemporalWindow that turns the
# state branch from single-frame retrieval into (prior, posterior) sequence pairs.
# Outcome: `rollout_timedelta` and `state_window` are defined; `state_window` is a
# PET PipelineIndex step that, given a reference date, returns
# ([state(t-dt), state(t)], [state(t+dt)]).
# -----------------------------------------------------------------------------

# Monthly cadence, matching time_interval = '1MS' used for normalisation above.
rollout_timedelta = petdata.time.TimeDelta((1, "month"))

# prior_indexes=[-1, 0]  -> retrieves state at (t-dt, t): two most recent states,
#                           analogous to using model time tendencies in PDE-based
#                           integration (Samudra Sec 2.2).
# posterior_indexes=[1]  -> retrieves state at (t+dt): the single-step prediction
#                           target. Multi-step rollout during training is handled
#                           separately, below, by re-querying the pipeline at
#                           successive reference dates rather than by widening this
#                           window -- see the rollout-loss section.
#
# merge_method concatenates each retrieved group along the time axis (functools.partial
# pattern taken directly from PyEarthTools' own TemporalWindowDemo.ipynb).
state_window = petpipe.modifications.TemporalWindow(
    prior_indexes=[-1, 0],
    posterior_indexes=[1],
    timedelta=rollout_timedelta,
    merge_method=functools.partial(np.concatenate, axis=0),
)


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: assemble the full forward-emulation pipeline. Each branch is a
# complete source pipeline so index-aware steps such as TemporalWindow receive
# the queried date instead of an already-retrieved sample.
# Outcome: `pipeline_i[date]` returns a nested tuple
#   ( (prior_states, posterior_states), forcing_at_t )
# where prior_states has shape (2, 1, H, W), posterior_states has shape
# (1, 1, H, W), and forcing_at_t has shape (1, 1, H, W).
# -----------------------------------------------------------------------------

state_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *state_steps,
    state_window,
)

forcing_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *forcing_steps,
)

pipeline_i = petpipe.Pipeline(
    petpipe.branching.PipelineBranchPoint(
        state_pipeline,    # branch 0: windowed OHC state
        forcing_pipeline,  # branch 1: flux forcing, single timestep
    ),
    # Start one month after time_start because state_window uses prior_indexes=[-1, 0].
    iterator=petpipe.iterators.DateRange("2000-02", time_end, interval="1 month"),
)


## Define the training, skill-test, and control-rollout splits

Samudra's evaluation protocol uses two separate long rollouts, deliberately kept apart
because they isolate different failure modes:

1. An **8-year skill-test rollout** against real, held-out atmospheric/surface forcing,
   used to measure how well the emulator tracks genuine forced trends.
2. A **century-scale control rollout** against *repeated* forcing from a window chosen
   specifically for near-zero net heat flux. This isolates pure equilibrium/numerical
   drift from drift caused by a genuinely changing forcing signal.

Conflating these into a single long rollout against real forcing would make it
impossible to tell whether divergence from ground truth reflects the model failing to
track a real trend, or the model drifting under flat forcing -- so both are defined
explicitly below, using PET's native `DateRange` and `SuperIterator`.

The training split itself is unchanged in spirit from `AutoEncoder_om2.ipynb` -- same
PET `DateRange` iterator class, same general boundary dates -- only the validation
window is trimmed slightly to leave room for the two held-out rollout regimes.


In [ ]:
# We define the training / validation / skill-test / control-rollout splits here.
#
# `train_split`   -- used for gradient updates. Starts at 2000-02 so the
#                    one-month prior state requested by state_window exists.
# `valid_split`   -- used for early stopping / monitoring during training, as before.
# `skill_test_split` -- held-out window with REAL forcing, for measuring trend-tracking
#                        skill via an autoregressive rollout (Samudra Sec 2.5, "8-year
#                        rollout").
# `control_window`   -- a single near-zero-net-flux window, chosen the same way as
#                        Samudra's control run, to be REPEATED via SuperIterator below
#                        rather than iterated once.

splits = {
    "train_split": petpipe.iterators.DateRange(
        "2000-02", "2014-01", interval="1 month"
    ),
    "valid_split": petpipe.iterators.DateRange(
        "2014-01", "2015-04", interval="1 month"
    ),
}

# Held-out skill-test window: real forcing, used for an autoregressive rollout
# (see the "Skill-test rollout" section near the end of this notebook). Kept
# separate from `splits` since it is not used for training-time gradient updates
# or validation-loss monitoring -- it is consumed directly by the rollout loop.
skill_test_split = petpipe.iterators.DateRange(
    "2015-04", "2019-01", interval="1 month"
)

# Control-rollout window: a single decade chosen for near-zero net surface heat
# flux, the same rationale as Samudra Sec 2.5. The actual near-zero-flux window
# for this dataset should be identified empirically from `total_surface_heat_flx`
# (e.g. by inspecting its area-weighted annual mean over the training period) --
# the date range below is a placeholder to be confirmed against the real data
# before the control rollout is run.
control_window = petpipe.iterators.DateRange(
    "2003-01", "2004-01", interval="1 month"
) # We use RYF0304 from Stewart et al., 2020

# Repeat the control window N times via SuperIterator, chaining the same
# DateRange back-to-back -- this is the PET-native equivalent of Samudra's
# "repeat 10-year cycle" control protocol, built from the existing DateRange
# class rather than a bespoke repeating-iterator wrapper.
n_control_repeats = 100  # 100 repeats of a 1-year window = a century-scale rollout,
                         # matching the order of magnitude of Samudra's century run.
control_iterator = petpipe.iterators.SuperIterator(
    *([control_window] * n_control_repeats)
)


## Define the forward-emulator architecture

The base `UNet` and `PartialConv2d` land-handling from `Emulator.py` are kept as-is.
This notebook only wraps the UNet with the forward-emulation channel contract:
two prior OHC states plus one forcing channel in, one future OHC state out.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: thin wrapper around the existing UNet, rewiring channel counts for
# the forward-emulation contract (2 past states + forcing in, 1 future state out)
# and handling the reshape from the pipeline's nested output structure.
# -----------------------------------------------------------------------------

class ForwardUNet(nn.Module):
    """
    Forward one-step OHC emulator.

    Wraps the existing UNet from Emulator.py unchanged at the block level; only
    the input/output channel counts differ from the reconstruction model in
    AutoEncoder_om2.ipynb. PartialConv2d land-handling is inherited from the base
    UNet implementation and is not modified here.

    forward() expects:
        prior_states : (B, 2, H, W)  -- OHC(t-dt), OHC(t), channel-stacked
        forcing      : (B, 1, H, W)  -- flux(t)
    and returns:
        next_state   : (B, 1, H, W)  -- predicted OHC(t+dt)
    """

    def __init__(self, n_prior_states: int = 2, n_forcing_channels: int = 1):
        super().__init__()
        self.n_prior_states = n_prior_states
        self.n_forcing_channels = n_forcing_channels
        self.base_unet = UNet(
            input_channel_count=n_prior_states + n_forcing_channels,
            output_channel_count=1,
        )

    def forward(self, prior_states: torch.Tensor, forcing: torch.Tensor) -> torch.Tensor:
        x = torch.cat([prior_states, forcing], dim=1)
        return self.base_unet(x)

## Multi-step rollout loss

Single-step masked MSE, as used in `AutoEncoder_om2.ipynb`, is replaced with a
multi-step recurrent rollout loss. The model's own predictions are fed back into
the next step, so compounding autoregressive error is visible during training.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: multi-step recurrent rollout loss for the deterministic ForwardUNet.
# Replaces the single-step masked MSE from AutoEncoder_om2.
# -----------------------------------------------------------------------------

def rollout_loss(
    model,
    initial_prior_states,
    forcing_sequence,
    target_sequence,
    mask,
    n_steps=4,
):
    """
    Multi-step recurrent rollout loss.

    The model's own predictions are fed back as input for n_steps passes,
    accumulating masked MSE over every step in the rollout window.
    """
    prior_states = initial_prior_states
    mse_total = 0.0

    for step in range(n_steps):
        forcing_t = forcing_sequence[:, step : step + 1]
        target_t = target_sequence[:, step : step + 1]

        pred_t = model(prior_states, forcing_t)
        step_err = (pred_t - target_t) ** 2
        step_mse = (step_err * mask).sum() / mask.sum().clamp_min(1.0)
        mse_total = mse_total + step_mse

        # Feed prediction back in as context for the next step.
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

    return mse_total


## Lightning wrapper for autoregressive training

`LightningWrapper` from `Emulator.py` was built for single-frame reconstruction and is
not reused directly here. `ForwardLightningWrapper` below carries over the same
Lightning conventions (constructor signature, Adam optimiser, masked loss) and adapts
the training step to unpack the nested pipeline output and call `rollout_loss`.

In [ ]:
class ForwardLightningWrapper(L.LightningModule):
    """
    Lightning wrapper for the deterministic autoregressive forward emulator.

    Expects each batch as ((prior_states, posterior_states), forcing). PET emits
    variables as a singleton channel axis, so `_step` squeezes `(B, T, 1, H, W)`
    tensors to the `(B, T, H, W)` contract used by `ForwardUNet`.
    """

    def __init__(self, model, mask, lr=1e-4, n_steps=1):
        super().__init__()
        self.model = model
        self.register_buffer("mask", torch.as_tensor(mask, dtype=torch.float32))
        self.lr = lr
        self.n_steps = n_steps

    @staticmethod
    def _squeeze_variable_axis(x):
        return x.squeeze(2) if x.ndim == 5 and x.shape[2] == 1 else x

    def _step(self, batch):
        (prior_states, posterior_states), forcing = batch
        prior_states = self._squeeze_variable_axis(prior_states)
        posterior_states = self._squeeze_variable_axis(posterior_states)
        forcing = self._squeeze_variable_axis(forcing)
        return rollout_loss(
            model=self.model,
            initial_prior_states=prior_states,
            forcing_sequence=forcing,
            target_sequence=posterior_states,
            mask=self.mask,
            n_steps=self.n_steps,
        )

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.lr)


In [ ]:
base_model = ForwardUNet(n_prior_states=2, n_forcing_channels=1)

lightning_model = ForwardLightningWrapper(
    model=base_model,
    mask=mask.values,
    lr=1e-4,
    n_steps=1,
)


## Data module and fast dataloaders

`PipelineLightningDataModule` wraps `pipeline_i`. Because this forward-emulator
pipeline returns nested samples of the form `((prior_states, posterior_states),
forcing)`, this notebook uses a local cached dataloader helper that preserves that
nested structure instead of the single-tensor `make_fast_dl` helper from the
autoencoder notebook.


In [ ]:
class RolloutTensorDataset(Dataset):
    def __init__(self, prior_states, posterior_states, forcing):
        self.prior_states = prior_states
        self.posterior_states = posterior_states
        self.forcing = forcing

    def __len__(self):
        return self.prior_states.shape[0]

    def __getitem__(self, idx):
        return (
            (self.prior_states[idx], self.posterior_states[idx]),
            self.forcing[idx],
        )


def make_fast_rollout_dl(pet_dl, batch_size=8, shuffle=False, drop_last=False):
    prior_batches = []
    posterior_batches = []
    forcing_batches = []

    for batch in pet_dl:
        (prior_states, posterior_states), forcing = batch
        prior_batches.append(prior_states.detach().cpu())
        posterior_batches.append(posterior_states.detach().cpu())
        forcing_batches.append(forcing.detach().cpu())

    dataset = RolloutTensorDataset(
        torch.cat(prior_batches, dim=0),
        torch.cat(posterior_batches, dim=0),
        torch.cat(forcing_batches, dim=0),
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        drop_last=drop_last,
    )


datamodule = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    pipeline_i,
    **splits,
    batch_size=32,
    num_workers=0,
)


In [ ]:
%%time
datamodule.setup("fit")

fast_train_dl = make_fast_rollout_dl(
    datamodule.train_dataloader(),
    batch_size=32,
    shuffle=False,
    drop_last=False,
)

fast_valid_dl = make_fast_rollout_dl(
    datamodule.val_dataloader(),
    batch_size=32,
    shuffle=False,
    drop_last=False,
)

## Trainer

Unchanged from `AutoEncoder_om2.ipynb` -- same `L.Trainer` configuration and same
reason for bypassing PET's native `pyearthtools.training.lightning.Train` (too slow;
GitHub issue linked in the original notebook).

In [ ]:
# PET's native training (pyearthtools.training.lightning.Train) is too slow
# for this setup -- see https://github.com/ACCESS-Community-Hub/PyEarthTools/issues/265

trainer = L.Trainer(
    max_epochs=200,
    num_sanity_val_steps=0,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

trainer.fit(
    model=lightning_model,
    train_dataloaders=fast_train_dl,
    val_dataloaders=fast_valid_dl,
)


## Skill-test rollout: held-out real forcing

An autoregressive rollout against the held-out `skill_test_split` window, using real
surface flux forcing. This measures whether the deterministic ForwardUNet tracks
genuine forced OHC trends over the test period. The rollout starts from one
ground-truth initial condition, then feeds model predictions back into the next
step without injecting ground-truth OHC again.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: autoregressive skill-test rollout against real held-out forcing.
# Runs the trained model forward from a single initial condition, supplying
# real flux at each step. Stores predictions and ground-truth for diagnostics.
# -----------------------------------------------------------------------------

base_model.eval()
base_model.to(device)

# Build a skill-test pipeline over the held-out real-forcing window.
# Same branch-first structure as pipeline_i, but with skill_test_split as iterator.
skill_pipeline = petpipe.Pipeline(
    petpipe.branching.PipelineBranchPoint(
        state_pipeline,
        forcing_pipeline,
    ),
    iterator=skill_test_split,
)

skill_preds = []
skill_targets = []

with torch.no_grad():
    prior_states = None
    for date in skill_test_split:
        (prior_window, target_window), forcing_window = skill_pipeline[date]
        # On the first step, seed from ground truth.
        if prior_states is None:
            prior_states = torch.tensor(
                prior_window, dtype=torch.float32, device=device
            ).unsqueeze(0).squeeze(2)

        forcing_t = torch.tensor(
            forcing_window, dtype=torch.float32, device=device
        ).unsqueeze(0).squeeze(2)
        target_t = torch.tensor(
            target_window, dtype=torch.float32, device=device
        ).unsqueeze(0).squeeze(2)

        pred_t = base_model(prior_states, forcing_t)

        skill_preds.append(pred_t.cpu().numpy())
        skill_targets.append(target_t.cpu().numpy())

        # Autoregressive: feed prediction back in, not ground truth.
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

skill_preds = np.concatenate(skill_preds, axis=0)
skill_targets = np.concatenate(skill_targets, axis=0)
print(f'Skill-test rollout complete. Prediction array: {skill_preds.shape}')


## Control rollout: repeated flat forcing (isolates equilibrium drift)

An autoregressive rollout driven by the repeated, near-zero-net-flux control window
(`control_iterator`), rather than real forcing. By running the model under forcing
that carries minimal net heat input, long-term drift in OHC is interpreted as
compounding emulator bias rather than a response to a real forced trend.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: long control rollout under repeated near-zero-net-flux forcing,
# to isolate equilibrium drift from genuine forced trend response.
# control_iterator chains the same 10-year DateRange n_control_repeats times
# via SuperIterator -- see the splits section for the rationale.
# -----------------------------------------------------------------------------

# Build a control pipeline using the repeated flat-forcing iterator.
control_pipeline = petpipe.Pipeline(
    petpipe.branching.PipelineBranchPoint(
        state_pipeline,
        forcing_pipeline,
    ),
    iterator=control_iterator,
)

control_preds = []

with torch.no_grad():
    prior_states = None
    for date in control_iterator:
        (prior_window, _), forcing_window = control_pipeline[date]
        if prior_states is None:
            prior_states = torch.tensor(
                prior_window, dtype=torch.float32, device=device
            ).unsqueeze(0).squeeze(2)

        forcing_t = torch.tensor(
            forcing_window, dtype=torch.float32, device=device
        ).unsqueeze(0).squeeze(2)

        pred_t = base_model(prior_states, forcing_t)

        control_preds.append(pred_t.cpu().numpy())
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

control_preds = np.concatenate(control_preds, axis=0)
print(f'Control rollout complete. Array shape: {control_preds.shape}')


## Drift diagnostics

Two complementary diagnostics, matching Samudra's evaluation framework:

1. **Skill-test trend attenuation** under real held-out forcing. Compare the
   predicted global-mean OHC trend with the OM2 target trend.
2. **Control-rollout equilibrium drift** under repeated near-zero-net-flux forcing.
   A clean baseline should not create a large artificial century-scale trend when
   the repeated forcing cycle is close to balanced.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: compute and tabulate the two core drift diagnostics.
# Un-normalise both rollouts back to physical OHC units using the same
# normalisation parameters used during training.
# -----------------------------------------------------------------------------

# Un-normalise back to physical units (J/m^2).
# Normalisation was Spatial_climatology, so undo: x_phys = x_norm * deviation + mean
ohc_mean = mean['ocean_heat_content_2d'].values[np.newaxis, :, :]
ohc_std = deviation['ocean_heat_content_2d'].values[np.newaxis, :, :]
mask_np = mask.values  # (H, W), 1=ocean 0=land

# skill_preds/skill_targets both shaped (T, 1, H, W) -> squeeze to (T, H, W)
pred_phys = skill_preds[:, 0] * ohc_std + ohc_mean
target_phys = skill_targets[:, 0] * ohc_std + ohc_mean
ctrl_phys = control_preds[:, 0] * ohc_std + ohc_mean

# Area-weighted global mean (mask acts as area weight proxy at 1-degree resolution)
ocean_cells = mask_np.sum()
pred_global = (pred_phys * mask_np).sum(axis=(1, 2)) / ocean_cells
target_global = (target_phys * mask_np).sum(axis=(1, 2)) / ocean_cells
ctrl_global = (ctrl_phys * mask_np).sum(axis=(1, 2)) / ocean_cells

# Linear trend fits
T_skill = len(pred_global)
T_ctrl = len(ctrl_global)
t_skill = np.arange(T_skill)
t_ctrl = np.arange(T_ctrl)

def lin_trend(t, y):
    p = np.polyfit(t, y, 1)
    return p[0]  # slope per timestep (here: per month)

pred_trend = lin_trend(t_skill, pred_global)
target_trend = lin_trend(t_skill, target_global)
ctrl_trend = lin_trend(t_ctrl, ctrl_global)
ctrl_std = ctrl_global.std()

attenuation_ratio = pred_trend / target_trend if target_trend != 0 else float('nan')

print('--- Drift diagnostics ---')
print(f'Skill-test | OM2 trend:      {target_trend:.4e} J/m^2/month')
print(f'Skill-test | Emulator trend: {pred_trend:.4e} J/m^2/month')
print(f'Skill-test | Attenuation:    {attenuation_ratio:.3f} (1.0 = perfect, <1 = damped)')
print()
print(f'Control    | Drift trend:    {ctrl_trend:.4e} J/m^2/month (should be ~0)')
print(f'Control    | Std dev:        {ctrl_std:.4e} J/m^2 (should be stable)')


## Plots

Two plot panels mirroring the evaluation structure above.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: skill-test rollout plot -- global mean OHC trajectory,
# emulator vs OM2 ground truth, with linear trend lines.
# -----------------------------------------------------------------------------

skill_time = np.arange(len(pred_global))
pred_trend_line = np.polyval(np.polyfit(skill_time, pred_global, 1), skill_time)
true_trend_line = np.polyval(np.polyfit(skill_time, true_global, 1), skill_time)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(true_global, label='OM2 target', color='black')
ax.plot(pred_global, label='ForwardUNet', color='tomato', linestyle='--')
ax.plot(true_trend_line, color='black', alpha=0.4)
ax.plot(pred_trend_line, color='tomato', alpha=0.4)
ax.set_ylabel('Global mean OHC')
ax.set_xlabel('Monthly rollout step')
ax.set_title('Held-out real-forcing skill-test rollout')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: control rollout plot -- long-term global mean OHC trajectory
# under repeated near-zero-net-flux forcing.
# -----------------------------------------------------------------------------

control_time = np.arange(len(control_global))
control_trend_line = np.polyval(
    np.polyfit(control_time, control_global, 1), control_time
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(control_global, label='ForwardUNet control rollout', color='navy')
ax.plot(control_trend_line, label='Linear drift trend', color='navy', alpha=0.4)
ax.set_ylabel('Global mean OHC')
ax.set_xlabel('Monthly rollout step')
ax.set_title(
    f'Control rollout | {n_control_repeats} repeats of {len(control_window)}-step window'
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## Closing note

This notebook now runs one clean deterministic baseline: `ForwardUNet` with
multi-step recurrent masked MSE. The skill-test and control-rollout diagnostics
are intended to characterise that baseline before adding any extra modelling
complexity elsewhere.
